In [ ]:
# start Docker-Container (with your own path to the folder)
! docker run --name neo4j-graphrag -p 7474:7474 -p 7687:7687 -d -v <path-to-folder>/neo4j/data:/data -v <path-to-folder>/neo4j/plugins:/plugins --env NEO4J_AUTH=neo4j/your-password neo4j:latest

In [21]:
! uv pip install langchain langchain-community langchain-neo4j langchain-openai
! uv pip install neo4j 
! uv pip install tiktoken python-dotenv
! uv pip install langchain-text-splitters
! uv pip install wikipedia
! uv pip install langchain_experimental
#neo4j-graphrag-python

Using Python 3.12.12 environment at: C:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv
Audited 4 packages in 250ms
Using Python 3.12.12 environment at: C:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv
Audited 1 package in 31ms
Using Python 3.12.12 environment at: C:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv
Audited 2 packages in 33ms
Using Python 3.12.12 environment at: C:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv
Audited 1 package in 39ms
Using Python 3.12.12 environment at: C:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv
Audited 1 package in 37ms
Using Python 3.12.12 environment at: C:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv
Resolved 48 packages 

In [2]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

In [5]:
load_dotenv()
os.environ["OPENAI_API_KEY"] = "None"
os.environ["NEO4J_URI"] = "neo4j://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "zJIhp8bq"
# Establish graph connection
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)
print("Connected to Neo4j successfully.")

Connected to Neo4j successfully.


In [17]:
from langchain_text_splitters import TokenTextSplitter
from langchain_community.document_loaders import WikipediaLoader

In [19]:
# Load documents (using Wikipedia as an example source)
raw_documents = WikipediaLoader(query="Artificial Intelligence").load()
# Chunk with overlap for context preservation
text_splitter = TokenTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
)
documents = text_splitter.split_documents(raw_documents)
print(f"Split into {len(documents)} chunks.")

Split into 49 chunks.


In [28]:
from openai import OpenAI
import instructor
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
# Initialize the LLM for extraction
#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("BASE_URL"))
#client = instructor.from_openai(client, mode=instructor.Mode.JSON)
llm = ChatOpenAI(model="Qwen3-30B-A3B-Instruct", temperature=0, base_url =os.getenv("BASE_URL"))
# Create the graph transformer
llm_transformer = LLMGraphTransformer(llm=llm)
# Extract graph documents (entities + relationships)
graph_documents = llm_transformer.convert_to_graph_documents(documents)
# Inspect what was extracted
for doc in graph_documents[:2]:
    print(f"Nodes: {doc.nodes}")
    print(f"Relationships: {doc.relationships}")
    print("---")
# Store in Neo4j
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,  # adds a generic __Entity__ label
    include_source=True,   # links graph nodes back to source chunks
)

c:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DynamicGraph(nodes=[Simpl... type='aim_to_ensure')]), input_type=DynamicGraph])
  return self.__pydantic_serializer__.to_python(
c:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DynamicGraph(nodes=[Simpl...task', type='enables')]), input_type=DynamicGraph])
  return self.__pydantic_serializer__.to_python(
c:\Users\qq6-xd4\Documents\Programmierung\llm_process_extraction_test\llm_process_extraction_test\.venv\Li

Nodes: [Node(id='Artificial Intelligence', type='Concept', properties={}), Node(id='Computational Systems', type='Concept', properties={}), Node(id='Human Intelligence', type='Concept', properties={}), Node(id='Learning', type='Concept', properties={}), Node(id='Reasoning', type='Concept', properties={}), Node(id='Problem-Solving', type='Concept', properties={}), Node(id='Perception', type='Concept', properties={}), Node(id='Decision-Making', type='Concept', properties={}), Node(id='Computer Science', type='Field', properties={}), Node(id='Google Search', type='Application', properties={}), Node(id='Youtube', type='Application', properties={}), Node(id='Amazon', type='Application', properties={}), Node(id='Netflix', type='Application', properties={}), Node(id='Google Assistant', type='Application', properties={}), Node(id='Siri', type='Application', properties={}), Node(id='Alexa', type='Application', properties={}), Node(id='Waymo', type='Application', properties={}), Node(id='Languag

In [29]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

In [43]:
# Create a hybrid (vector + keyword) index over the stored chunks
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(model = "qwen3_emb_8b_40k", base_url="http://hal9000.skim.th-owl.de:11976/v1", check_embedding_ctx_length=False), #os.getenv("BASE_URL")
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
# Test it
results = vector_index.similarity_search(
    "What are the major approaches to artificial intelligence?"
)
for r in results[:3]:
    print(r.page_content[:200])
    print("---")


text: The following outline is provided as an overview of and topical guide to artificial intelligence:
Artificial intelligence (AI) is intelligence exhibited by machines or software. It is also the 
---

text:  Kingdom started a series of international summits on AI with the AI Safety Summit. It was followed by the AI Seoul Summit in 2024, and the AI Action Summit in Paris in 2025.


== Perspectives 
---

text: This glossary of artificial intelligence is a list of definitions of terms and concepts relevant to the study of artificial intelligence (AI), its subdisciplines, and related fields. Related gl
---


In [44]:
from langchain_neo4j import GraphCypherQAChain

In [49]:
# The Cypher QA chain lets the LLM generate graph queries dynamically
cypher_chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(model="qwen3_8b_40k", temperature=0, base_url ="http://hal9000.skim.th-owl.de:11976/v1"),
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,  # required for dynamic Cypher
)
# Ask a relationship-based question
response = cypher_chain.invoke({
    "query": "What technologies are related to machine learning?"
})
print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Technology)-[:RELATED_TO]->(ml:MachineLearning)
RETURN t.name


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `MachineLearning` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=41, offset=40>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 40, 'line': 1, 'column': 41}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (t:Technology)-[:RELATED_TO]->(ml:MachineLearning)\nRETURN t.name'


Full Context:
[]

> Finished chain.
I don't know the answer.
